# Cuaderno 03: Cuando una Demostración se Atasca — Generalización de Invariantes (`generalizing`)

**Objetivo:** Reproducir en vivo el bloqueo clásico al intentar verificar por inducción directa un algoritmo recursivo de cola con acumulador (`fastRev`), inspeccionar el estado bloqueado del Infoview y resolverlo con `induction ... generalizing acc`.

In [1]:
import pathlib, shutil, subprocess, tempfile, time

def find_lean() -> str:
    elan = shutil.which("elan") or str(pathlib.Path.home() / ".elan" / "bin" / "elan")
    if pathlib.Path(elan).is_file():
        out = subprocess.check_output([elan, "which", "lean"], text=True).strip()
        if out:
            return out
    lean = shutil.which("lean")
    if lean:
        return lean
    raise RuntimeError("No se encontró `lean` (instala elan desde https://lean-lang.org).")

LEAN_BIN = find_lean()

def run_lean(code: str, title: str = "", expect_error: bool = False) -> str:
    """Compila código Lean 4 real y muestra los estados `trace_state`, `#eval`, `#print` o errores."""
    t0 = time.perf_counter()
    with tempfile.NamedTemporaryFile("w", suffix=".lean", encoding="utf-8", delete=False) as f:
        f.write(code.strip() + "\n")
        tmp_path = pathlib.Path(f.name)
    try:
        proc = subprocess.run([LEAN_BIN, str(tmp_path)], capture_output=True, text=True, timeout=45)
        dt = (time.perf_counter() - t0) * 1000.0
        out = ((proc.stdout or "") + (proc.stderr or "")).replace(str(tmp_path), "snippet.lean").strip()
        status = "ERROR ESPERADO (DIAGNÓSTICO)" if (proc.returncode != 0 and expect_error) else (
                 "VERIFICADO POR EL KERNEL" if proc.returncode == 0 else "FALLO DE COMPILACIÓN")
        header = f"=== [{status}] {title} ({dt:.0f} ms) ==="
        print(header)
        print(out if out else "(Compilación limpia sin mensajes adicionales: 0 errores, 0 warnings)")
        print()
        if proc.returncode != 0 and not expect_error:
            raise RuntimeError(f"Lean falló inesperadamente en {title}")
        if proc.returncode == 0 and expect_error:
            raise RuntimeError(f"Se esperaba un fallo pedagógico en {title}, pero compiló con éxito")
        return out
    finally:
        tmp_path.unlink(missing_ok=True)

print("Compilador Lean activo:", LEAN_BIN)

Compilador Lean activo: /Users/legalintermedia/.elan/toolchains/leanprover--lean4---v4.36.0/bin/lean


## 1. Reproducción en Vivo del Bloqueo Inductivo (`expect_error=True`)
Intentamos demostrar `revAcc xs [] = xs.reverse` haciendo `induction xs` directamente. Observa en la salida del compilador cómo `ih` queda atada a `[]` mientras la meta pide `revAcc rest [x]`.

In [2]:
stuck_code = """
def revAcc {α : Type} : List α → List α → List α
  | [],      acc => acc
  | x :: xs, acc => revAcc xs (x :: acc)

theorem intento_bloqueado {α : Type} (xs : List α) :
    revAcc xs [] = xs.reverse := by
  induction xs with
  | nil => rfl
  | cons x rest ih =>
    dsimp [revAcc]
    trace_state -- ¡Observa el desajuste entre `ih` (acumulador `[]`) y la meta (`[x]`)!
"""
_ = run_lean(stuck_code, "Demostración bloqueada sin generalizar el acumulador", expect_error=True)

=== [ERROR ESPERADO (DIAGNÓSTICO)] Demostración bloqueada sin generalizar el acumulador (269 ms) ===
snippet.lean:9:19: error: unsolved goals
case cons
α : Type
x : α
rest : List α
ih : revAcc rest [] = rest.reverse
⊢ revAcc rest [x] = (x :: rest).reverse
case cons
α : Type
x : α
rest : List α
ih : revAcc rest [] = rest.reverse
⊢ revAcc rest [x] = (x :: rest).reverse



## 2. Desbloqueo Mediante Invariante General + `induction xs generalizing acc`
Generalizamos `[]` a un acumulador arbitrario `acc` enunciando `revAcc xs acc = xs.reverse ++ acc`, y cuantificamos `ih` universalmente con `generalizing acc`.

In [3]:
fixed_code = """
def revAcc {α : Type} : List α → List α → List α
  | [],      acc => acc
  | x :: xs, acc => revAcc xs (x :: acc)

def fastRev {α : Type} (xs : List α) : List α := revAcc xs []

theorem revAcc_invariant {α : Type} (xs acc : List α) :
    revAcc xs acc = xs.reverse ++ acc := by
  induction xs generalizing acc with
  | nil => simp [revAcc]
  | cons x rest ih =>
    trace_state -- ¡Ahora `ih : ∀ (acc : List α), revAcc rest acc = rest.reverse ++ acc`!
    calc
      revAcc (x :: rest) acc
        = revAcc rest (x :: acc)       := rfl
      _ = rest.reverse ++ (x :: acc)   := ih (x :: acc)
      _ = (rest.reverse ++ [x]) ++ acc := by simp [List.append_assoc]
      _ = (x :: rest).reverse ++ acc   := by simp [List.reverse_cons]

theorem fastRev_correct {α : Type} (xs : List α) : fastRev xs = xs.reverse := by
  dsimp [fastRev]
  rw [revAcc_invariant, List.append_nil]

#eval fastRev [10, 20, 30, 40, 50]
#print axioms fastRev_correct
"""
_ = run_lean(fixed_code, "Invariante generalizado con `generalizing acc`")

=== [VERIFICADO POR EL KERNEL] Invariante generalizado con `generalizing acc` (303 ms) ===
case cons
α : Type
x : α
rest : List α
ih : ∀ (acc : List α), revAcc rest acc = rest.reverse ++ acc
acc : List α
⊢ revAcc (x :: rest) acc = (x :: rest).reverse ++ acc
[50, 40, 30, 20, 10]
'fastRev_correct' depends on axioms: [propext]



### Q&A
* **¿Cómo se detecta en el Infoview que falta generalizar una hipótesis inductiva?** Cuando la llamada recursiva en la meta tiene un argumento modificado (como `[x]` o `x :: acc`), mientras que en `ih` ese mismo argumento aparece fijo como una constante (`[]` o `0`).
* **¿Qué hace exactamente `induction xs generalizing acc`?** Revierte `acc` antes de aplicar el principio de inducción sobre `xs`, transformando `ih` en una función `∀ (acc : List α), ...` aplicable a cualquier nuevo acumulador `ih (x :: acc)`.

### Data Analysis Key Findings
* En el intento ingenuo, el compilador reportó `unsolved goals` porque `ih : revAcc rest [] = rest.reverse` no puede reescribir `revAcc rest [x]`.
* Al introducir el invariante `revAcc xs acc = xs.reverse ++ acc` con `generalizing acc`, tanto el invariante como el teorema principal `fastRev_correct` compilaron en **2 líneas de corolario** y `0` axiomas externos.

### Insights or Next Steps
* Aplica sistemáticamente `generalizing` siempre que verifiques máquinas abstractas (entornos `env`, pilas `stack`), plegados `foldl` o protocolos con estado cambiante.
* Pasa al **Cuaderno 04** para observar cómo un agente de IA ejecuta automáticamente este ciclo de diagnóstico y auto-reparación.